# 08 - Reporting
Generate automated reports and visualizations from all processed data.

In [ ]:
# ===== CONFIGURATION =====
GITHUB_REPO_URL = "https://github.com/kaarthik-balakrishnan/LightningPoseTrack.git"
GIT_BRANCH = "main"

DRIVE_ROOT = "/content/drive/My Drive/PigBehavior"
DRIVE_POSE_OUTPUTS = f"{DRIVE_ROOT}/pose_outputs"
DRIVE_FEATURES = f"{DRIVE_ROOT}/features"
DRIVE_BEHAVIOR_OUTPUTS = f"{DRIVE_ROOT}/behavior_outputs"
DRIVE_REPORTS = f"{DRIVE_ROOT}/reports"
FPS = 30.0
DRIVE_FOLDER_ID = "1X_41ZW3HfwVeft2lPld3XNqXsdxRDIwb"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, sys
REPO_DIR = "/content/LightningPoseTrack"
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

In [ ]:
!pip install --quiet pandas numpy matplotlib seaborn pyarrow scipy imageio[ffmpeg]

In [ ]:
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm
from src.reports.daily_report import generate_html_report
from src.pose.clean_pose import clean_pose_df
from src.features.orientation import compute_heading

pose_dir = Path(DRIVE_POSE_OUTPUTS)
features_dir = Path(DRIVE_FEATURES)
behavior_dir = Path(DRIVE_BEHAVIOR_OUTPUTS)
reports_dir = Path(DRIVE_REPORTS)
reports_dir.mkdir(parents=True, exist_ok=True)

pose_files = list(pose_dir.rglob("*.parquet"))
print(f"Generating reports for {len(pose_files)} videos...")

for pose_file in tqdm(pose_files, desc="Generating reports"):
    rel = pose_file.relative_to(pose_dir)
    parts = str(rel).split("/")
    session = parts[0] if len(parts) > 1 else "unknown"
    import re
    cam_match = re.search(r"_cam(\d+)_", pose_file.name)
    camera = int(cam_match.group(1)) if cam_match else 0

    # Load pose
    pose_df = pd.read_parquet(pose_file)
    pose_df_clean = clean_pose_df(pose_df)
    pose_df_clean["heading_deg"] = compute_heading(pose_df_clean).apply(lambda x: x * 180 / 3.14159 if pd.notna(x) else float("nan"))

    # Load features (if exists)
    feat_path = features_dir / rel
    feat_path = feat_path.with_name(feat_path.stem.replace("_pose", "_features") + ".parquet")
    features_df = pd.read_parquet(feat_path) if feat_path.exists() else None

    # Load feeding events (if exists)
    feed_path = behavior_dir / rel
    feed_path = feed_path.with_name(feed_path.stem.replace("_pose", "_feeding_events") + ".parquet")
    feeding_events = pd.read_parquet(feed_path) if feed_path.exists() else None

    # Generate HTML report
    report_name = f"report_{session}_cam{camera}.html"
    report_path = reports_dir / report_name
    generate_html_report(
        pose_df=pose_df_clean,
        feeding_events=feeding_events,
        features_df=features_df,
        output_path=report_path,
        session=session,
        camera=camera,
        fps=FPS,
    )

print(f"\nReports saved to {reports_dir}/")

In [ ]:
# Generate a master summary report
feeding_master = behavior_dir / "feeding_events.parquet"
if feeding_master.exists():
    df = pd.read_parquet(feeding_master)
    print("=== MASTER SUMMARY ===")
    print(f"Total feeding events: {len(df)}")
    print(f"Total feeding duration: {df['duration_sec'].sum() / 60:.1f} min")
    print(f"Mean bout duration: {df['duration_sec'].mean():.1f}s")
    print(f"\nBy session:")
    summary = df.groupby('session').agg(
        events=('duration_sec', 'count'),
        total_min=('duration_sec', lambda x: f"{x.sum()/60:.1f}"),
        mean_duration_s=('duration_sec', 'mean'),
    )
    print(summary)

print(f"\nAll reports are in: {DRIVE_REPORTS}/")